# Base de dados da Marília → `base_dados_marilia.csv`

Dados históricos de armadilhas de *Aedes* em POA (2019–2023), organizados pela Marília, em CSVs anuais (`saida_2019.csv` … `saida_2023.csv`, separador `;`). Cada linha = uma inspeção de armadilha numa semana.

**Objetivo desta etapa:** (1) **concatenar** todos os anos numa única `base_dados_marilia.csv`; (2) **explorar** o schema e como ele **converge com a base da raspagem** (`base_armadilhas_concatenada.csv`), para depois — no `modelo1.ipynb` — juntar as duas. Enquanto a série histórica oficial (Prefeitura/SMS) não chega, jogamos com o que temos.

Construção **passo a passo**, um bloco por vez.

## Bloco 1 — concatenar os CSVs anuais da Marília

Lê os 5 `saida_*.csv` (mesmo schema de 15 colunas), empilha tudo num só DataFrame e adiciona `arquivo_origem` para rastreabilidade.

In [ ]:
import glob
import os
from pathlib import Path

import pandas as pd


# Acha a raiz do projeto (Meu_Projeto/) subindo até encontrar a pasta 'Raspagem'
# (evita caminho absoluto hard-coded).
def achar_raiz(marcador="Raspagem"):
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marcador).is_dir():
            return p
    raise FileNotFoundError(f"Pasta-raiz contendo '{marcador}/' nao encontrada a partir de {Path.cwd()}")


RAIZ = achar_raiz()
MARILIA_DIR = RAIZ / "Bases de dados" / "dados_marilia"

arquivos = sorted(glob.glob(str(MARILIA_DIR / "saida_*.csv")))

partes = []
for caminho in arquivos:
    df = pd.read_csv(caminho, sep=";", parse_dates=["Data Inicio", "Data Fim"])
    df["arquivo_origem"] = os.path.basename(caminho)
    partes.append(df)

df_marilia = pd.concat(partes, ignore_index=True)

print(f"{len(arquivos)} arquivos concatenados:", [os.path.basename(a) for a in arquivos])
print("linhas x colunas:", df_marilia.shape)
print("anos:", sorted(df_marilia["Ano"].unique().tolist()))
print("período:", df_marilia["Data Inicio"].min().date(), "->", df_marilia["Data Fim"].max().date())
print("\nlinhas por ano:")
print(df_marilia["Ano"].value_counts().sort_index().to_string())

## Bloco 2 — schema e amostra

Colunas, tipos e primeiras linhas.

In [ ]:
display(df_marilia.dtypes.to_frame("tipo"))
df_marilia.head()

## Bloco 3 — exploração rápida

Três coisas que importam para a convergência com a raspagem: (a) o **clima vem vazio**; (b) o **`ID` é único por linha** (inspeção), então a armadilha recorrente só se identifica por `(Latitude, Longitude)`; (c) o volume de inspeções por semana (proxy de armadilhas inspecionadas).

In [ ]:
# (a) Clima: as 3 colunas vêm 100% vazias (placeholders) -> clima virá do InfoDengue.
print("clima (nulos / total):")
for c in ["Precipitation", "Temperature", "Relative humidity"]:
    print(f"  {c}: {df_marilia[c].isna().sum()} / {len(df_marilia)}")

# (b) Identidade da armadilha
print("\nID distintos:", df_marilia["ID"].nunique(),
      "| == nº de linhas?", df_marilia["ID"].nunique() == len(df_marilia))
print("pares (lat,long) distintos (~armadilhas):",
      df_marilia[["Latitude", "Longitude"]].drop_duplicates().shape[0])
print("Local distintos (~bairros/regiões):", df_marilia["Local"].nunique())

# (c) Inspeções por semana (proxy de armadilhas inspecionadas)
insp = df_marilia.groupby(["Ano", "Semana"]).size()
print("\ninspeções por semana — média:", round(insp.mean(), 1),
      "| min:", insp.min(), "| max:", insp.max())
print("Aedes aegypti por inspeção — média:", round(df_marilia["Aedes aegypti"].mean(), 3),
      "| max:", int(df_marilia["Aedes aegypti"].max()))

## Bloco 4 — convergência Marília × raspagem

Compara as colunas das duas bases para planejar o join (que será feito no `modelo1.ipynb`).

**Mapeamento proposto (conceito → coluna):**

| Conceito | Marília | Raspagem (`base_armadilhas`) |
|---|---|---|
| inspeção (id único/linha) | `ID` | `id` / `inspection_id` |
| armadilha fixa | *(não tem; usar `Latitude`,`Longitude`)* | `trap_id` |
| bairro/região | `Local` (~63) | `neighborhood` |
| ano / semana epi | `Ano`, `Semana` | de `week` (`SS/AAAA`) |
| início da semana | `Data Inicio` | `data_coleta` (aprox.) |
| coordenadas | `Latitude`, `Longitude` | `latitude`, `longitude` |
| *Ae. aegypti* (total) | `Aedes aegypti` | `aedes_aegypti_femea` + `aedes_aegypti_macho` |
| *Ae. aegypti* fêmea | *(não separa sexo)* | `aedes_aegypti_femea` |
| *Ae. albopictus* | `Aedes albopictus` | `aedes_albopictus_femea` + `_macho` |
| *Culex sp* | `Culex sp` | `culex_sp_femea` + `_macho` |
| clima | `Precipitation`/`Temperature`/`Relative humidity` *(vazio)* | *(não tem; vem do InfoDengue)* |

Como a MosquiTRAP captura fêmeas (machos ~0 na raspagem), o `Aedes aegypti` total da Marília ≈ fêmeas → as duas fontes ficam comparáveis no índice **aegypti por armadilha** (semanal).

In [ ]:
BASE_RASPAGEM = RAIZ / "Bases de dados" / "juntar_arquivos_raspagem" / "output" / "base_armadilhas_concatenada.csv"
cols_raspagem = list(pd.read_csv(BASE_RASPAGEM, nrows=0).columns)

print(f"MARÍLIA  ({len(df_marilia.columns)} cols):", list(df_marilia.columns))
print()
print(f"RASPAGEM ({len(cols_raspagem)} cols):", cols_raspagem)

## Bloco 5 — salvar `base_dados_marilia.csv`

Grava a base concatenada (schema nativo da Marília + `arquivo_origem`) em `output/`. A harmonização com a raspagem fica para o `modelo1.ipynb`.

In [ ]:
OUTPUT_DIR = MARILIA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
destino = OUTPUT_DIR / "base_dados_marilia.csv"

df_marilia.to_csv(destino, index=False)
print("salvo em:", destino)
print("linhas x colunas:", df_marilia.shape)
print("tamanho:", round(destino.stat().st_size / 1_000_000, 2), "MB")